# Phase 1 — Image baseline (offline-submittable)

2.5D **ConvNeXt-Tiny** + gated attention MIL on the **58 labeled** studies.

- **From-scratch** weights (pipeline validation; weak score expected with n=58).
- Reports are **not** used at inference (competition rule).
- Kaggle scoring: **internet OFF** — package from attached Dataset `rsna-knee-code`.
- Daily: Cursor → Kaggle Jupyter Server ([docs/KAGGLE.md](../docs/KAGGLE.md)).
- After `src/` changes: `python scripts/sync_code_to_jupyter.py` (interactive) or `python scripts/publish_code_dataset.py` (Save & Run All). Submit: `python scripts/push_kaggle_kernel.py train`.

See [docs/PROJECT_LOG.md](../docs/PROJECT_LOG.md).


## Setup


In [1]:
from __future__ import annotations

import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Always import from /kaggle/working/src (synced from your PC) or local repo.
# python scripts/sync_code_to_jupyter.py  →  /kaggle/working/src + configs


def _src() -> Path:
    work = Path("/kaggle/working")
    work_src = work / "src"
    mount = Path("/kaggle/input/rsna-knee-code")
    if work.is_dir() and not (work_src / "rsna_knee").is_dir() and (mount / "src" / "rsna_knee").is_dir():
        shutil.copytree(mount / "src", work_src)
        if (mount / "configs").is_dir():
            shutil.copytree(mount / "configs", work / "configs", dirs_exist_ok=True)
    if (work_src / "rsna_knee").is_dir():
        return work_src
    for root in (Path.cwd(), Path.cwd().parent):
        src = root / "src"
        if (src / "rsna_knee").is_dir():
            return src
    raise FileNotFoundError(
        "rsna_knee is not on this kernel. From the repo on your PC:\n"
        "  python scripts/sync_code_to_jupyter.py\n"
        "(paste the VS Code Compatible URL, or set KAGGLE_JUPYTER_URL in .env)\n"
        "Then re-run this cell. Competition DICOMs still need Input attached."
    )


_SRC = _src()
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
REPO_ROOT = _SRC.parent

from rsna_knee.constants import TARGET_LABELS
from rsna_knee.data import (
    KneeStudyDataset,
    load_sample_submission,
    load_test_table,
    load_train_table,
    predictions_to_submission,
)
from rsna_knee.data.schema import labels_present_mask
from rsna_knee.models import build_model
from rsna_knee.training import (
    macro_roc_auc,
    predict_test_ensemble,
    prevalence_baseline_predictions,
    run_kfold_training,
)
from rsna_knee.utils.config import load_config
from rsna_knee.utils.paths import default_data_root, is_kaggle_kernel

ON_KAGGLE = is_kaggle_kernel()
CFG_NAME = "kaggle" if ON_KAGGLE else "default"
cfg = load_config(CFG_NAME)
DATA_ROOT = default_data_root()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count() if DEVICE.type == "cuda" else 0
ALLOW_RANDOM = bool(cfg["model"].get("allow_random_init", True))

print(f"Environment : {'Kaggle' if ON_KAGGLE else 'local'}")
print(f"Data root   : {DATA_ROOT}")
print(f"Package     : {REPO_ROOT}")
print(f"Device      : {DEVICE}  ({N_GPUS} GPU{'s' if N_GPUS != 1 else ''})")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  gpu[{i}]    : {props.name}  {props.total_memory / 1024**3:.1f} GiB")
print(f"Config      : {CFG_NAME}")
print(f"Init        : {'from-scratch' if ALLOW_RANDOM else 'pretrained'}")
print(f"Torch       : {torch.__version__}")
print(f"AMP / batch : {cfg['training'].get('amp', True)}  /  {cfg['training']['batch_size']}")


Environment : Kaggle
Data root   : /kaggle/input/competitions/rsna-knee-abnormality-detection
Package     : /kaggle/working
Device      : cuda  (2 GPUs)
  gpu[0]    : Tesla T4  14.6 GiB
  gpu[1]    : Tesla T4  14.6 GiB
Config      : kaggle
Init        : from-scratch
Torch       : 2.10.0+cu128
AMP / batch : True  /  4


## 1. Labeled-study sanity check

Only explicitly labeled studies are used for supervised training.


In [2]:
train = load_train_table(DATA_ROOT)
has_labels = labels_present_mask(train)
n_labeled = int(has_labels.sum())
print(f"Train studies : {len(train):,}")
print(f"Labeled       : {n_labeled:,} ({100 * has_labels.mean():.1f}%)")

volume_shape = tuple(cfg["data"]["volume_shape"])
max_series = int(cfg["data"].get("max_series", 3))
print(f"Volume shape  : {volume_shape}  |  max_series={max_series}")


Train studies : 4,407
Labeled       : 58 (1.3%)
Volume shape  : (16, 256, 256)  |  max_series=3


## 2. Prevalence baseline (submission floor)

Constant positive rates from labeled studies — validates submission format.


In [3]:
labeled = train.loc[has_labels]
label_mat = labeled[TARGET_LABELS].to_numpy(dtype=np.float32)
mask_mat = (~np.isnan(label_mat)).astype(np.float32)
label_mat = np.nan_to_num(label_mat, nan=0.0)

test = load_test_table(DATA_ROOT)
n_test = len(test)
prev_preds = prevalence_baseline_predictions(label_mat, mask_mat, n_test)
prev_df = predictions_to_submission(test["StudyInstanceUID"].tolist(), pd.DataFrame(prev_preds, columns=TARGET_LABELS))

rates = prev_preds[0]
print("Positive rates (labeled):")
for name, rate in zip(TARGET_LABELS, rates):
    print(f"  {name:30s} {rate:.3f}")

OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else (REPO_ROOT / "outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
prev_path = OUT_DIR / "submission_prevalence.csv"
prev_df.to_csv(prev_path, index=False)
print(f"Wrote {prev_path}")
display(prev_df.head())


Positive rates (labeled):
  acl_tear                       0.414
  mcl_tear                       0.155
  medial_meniscus_injury         0.448
  lateral_meniscus_injury        0.397
  medial_osteoarthritis          0.259
  lateral_osteoarthritis         0.190
  patellofemoral_osteoarthritis  0.362
  joint_effusion                 0.603
  synovitis                      0.466
  bakers_cyst                    0.207
  bone_contusion                 0.328
  fracture                       0.310
Wrote /kaggle/working/submission_prevalence.csv


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.413793,0.155172,0.448276,0.396552,0.258621,0.189655,0.362069,0.603448,0.465517,0.206897,0.327586,0.310345
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.413793,0.155172,0.448276,0.396552,0.258621,0.189655,0.362069,0.603448,0.465517,0.206897,0.327586,0.310345
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.413793,0.155172,0.448276,0.396552,0.258621,0.189655,0.362069,0.603448,0.465517,0.206897,0.327586,0.310345


## 3. Model smoke check (from-scratch)

No ImageNet Dataset required — Phase 1 validates train → `submission.csv`.


In [4]:
smoke = KneeStudyDataset(
    DATA_ROOT,
    split="train",
    labeled_only=True,
    volume_shape=volume_shape,
    max_series=max_series,
    cache=False,
    require_dicom=True,
)
print(f"Labeled dataset size: {len(smoke)}")
if len(smoke) == 0:
    raise RuntimeError("No labeled studies — check data root / CSVs.")

sample = smoke[0]
print(f"Sample image shape: {sample['image'].shape}  (S, 3, H, W)")
print(f"Labels: {sample['labels']}")

model = build_model(
    cfg["model"]["name"],
    allow_random_init=ALLOW_RANDOM,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params:,}")

with torch.no_grad():
    x = torch.from_numpy(sample["image"]).unsqueeze(0)
    logits = model(x)
print(f"Logits shape: {tuple(logits.shape)}")


Labeled dataset size: 58
Sample image shape: (48, 3, 256, 256)  (S, 3, H, W)
Labels: [0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0.]
Model params: 28,221,805
Logits shape: (1, 12)


## 4. K-fold training

Study-level folds on labeled studies. Local default: 1 epoch smoke. Kaggle: full `configs/kaggle.yaml`.

Default: **GPU study bank** (~2 GiB fp16 for 58 studies) + **batch 4 on one T4** — steadier VRAM than `DataParallel` with batch 2. Sync code before running: `python scripts/sync_code_to_jupyter.py`.


In [5]:
CKPT_DIR = Path(cfg["paths"]["checkpoint_dir"])
if not CKPT_DIR.is_absolute():
    CKPT_DIR = REPO_ROOT / CKPT_DIR
CKPT_DIR.mkdir(parents=True, exist_ok=True)

MAX_EPOCHS = int(cfg["training"]["max_epochs"])
# Keep local CPU runs short
if not ON_KAGGLE and DEVICE.type == "cpu":
    MAX_EPOCHS = min(MAX_EPOCHS, 1)
    print(f"Local CPU: capping max_epochs → {MAX_EPOCHS}")

train_result = run_kfold_training(
    DATA_ROOT,
    n_folds=int(cfg["training"]["n_folds"]),
    max_epochs=MAX_EPOCHS,
    batch_size=int(cfg["training"]["batch_size"]),
    learning_rate=float(cfg["training"]["learning_rate"]),
    volume_shape=volume_shape,
    max_series=max_series,
    checkpoint_dir=CKPT_DIR,
    seed=int(cfg["seed"]),
    allow_random_init=ALLOW_RANDOM,
    tta=bool(cfg["inference"].get("tta", False)),
    num_workers=int(cfg["data"].get("num_workers", 0)),
    model_name=cfg["model"]["name"],
    use_amp=bool(cfg["training"].get("amp", True)),
    gpu_cache=bool(cfg["training"].get("gpu_cache", True)),
    data_parallel=bool(cfg["training"].get("data_parallel", False)),
    slice_chunk=int(cfg["model"].get("slice_chunk", 16)),
)

print(f"OOF macro ROC-AUC: {train_result['overall_auc']:.4f}")
for fr in train_result["fold_results"]:
    print(f"  fold {fr.fold}: {fr.val_auc:.4f}  {fr.checkpoint_path}")


58 studies | 5 folds × 8 epochs | cuda (single-gpu) | batch 4 | gpu-cache | amp=True


cache volumes (cpu):   0%|          | 0/58 [00:00<?, ?it/s]

Uploading cached volumes to GPU...
  GPU study bank: 1044 MiB  dtype=torch.float16


fold 0 epoch 1/8:   0%|          | 0/12 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 36.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 32.81 MiB is free. Including non-PyTorch memory, this process has 14.53 GiB memory in use. Of the allocated memory 14.38 GiB is allocated by PyTorch, and 15.07 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 5. Out-of-fold metric detail


In [ ]:
oof_preds = train_result["oof_preds"]
oof_labels = train_result["oof_labels"]
print("OOF macro ROC-AUC:", macro_roc_auc(oof_labels, oof_preds))

per_label = []
for i, name in enumerate(TARGET_LABELS):
    yt = oof_labels[:, i]
    if len(np.unique(yt)) < 2:
        per_label.append({"label": name, "auc": float("nan")})
        continue
    from sklearn.metrics import roc_auc_score
    per_label.append({"label": name, "auc": float(roc_auc_score(yt, oof_preds[:, i]))})
display(pd.DataFrame(per_label).sort_values("auc", ascending=False))


## 6. Test inference → submission.csv

Ensemble fold checkpoints (+ TTA on Kaggle). Writes `/kaggle/working/submission.csv`.


In [ ]:
ckpt_paths = [fr.checkpoint_path for fr in train_result["fold_results"]]
study_ids, test_preds = predict_test_ensemble(
    ckpt_paths,
    DATA_ROOT,
    volume_shape=volume_shape,
    max_series=max_series,
    batch_size=int(cfg["training"]["batch_size"]),
    tta=bool(cfg["inference"].get("tta", False)),
    allow_random_init=True,
    model_name=cfg["model"]["name"],
    num_workers=int(cfg["data"].get("num_workers", 0)),
    use_amp=bool(cfg["training"].get("amp", True)),
    data_parallel=bool(cfg["training"].get("data_parallel", False)),
    slice_chunk=int(cfg["model"].get("slice_chunk", 16)),
)

sub = predictions_to_submission(
    study_ids,
    pd.DataFrame(test_preds, columns=TARGET_LABELS),
)
sub_path = OUT_DIR / "submission.csv"
sub.to_csv(sub_path, index=False)
print(f"Wrote {sub_path}  shape={sub.shape}")
display(sub.head())

# Schema check vs sample submission
sample = load_sample_submission(DATA_ROOT)
assert list(sub.columns) == list(sample.columns), (list(sub.columns), list(sample.columns))
print("submission.csv columns match sample_submission.csv")
